In [21]:
#Import Modules
import arcpy
import arcgis
import datetime
from shutil import copyfile
import os
import sys
#import getpass
import xlrd, xlwt # I think we should be using openpyxl to create xlsx
import openpyxl
from openpyxl.utils import get_column_letter # can only run openpyxl in arcpro
from openpyxl.styles import Font
import pandas as pd
from openpyxl.chart import BarChart, Series, Reference
import matplotlib.pyplot as plt
import sys
import subprocess
from itertools import islice

In [22]:
########## PARAMETER INPUTS ##############
#AnalysisPoints = arcpy.GetParameterAsText(0) # this should be a points from step 2
AnalysisPoints = r'C:\Users\alaurencetraynor\Documents\2021\Analysis\UT\PHMA\parker_mtn_phma.gdb\parker_mtn_phma_AIM_points'
#configFile = arcpy.GetParameterAsText(1) # benchmark config file aka your monitoring objectives. Eventually this should pull from the first couple arctoolbox tools
configFile = r'C:\Users\alaurencetraynor\Documents\2021\Analysis\UT\PHMA\benchmarktooltest_UT_GRSG.xlsx'
#OutputFolder = arcpy.GetParameterAsText(2) # directory to store all outputs
OutputFolder = r'C:\Users\alaurencetraynor\Documents\2021\Analysis\UT\PHMA\Point Counting Results'

In [23]:
########## OUTPUT PATHS ##################
###### define these based on output folder
outExcel = OutputFolder + "\\" + "benchmarked_points.xlsx" # this excel should eventually have colnames which match up with aim.analysis inputs/benchmark tool
outGDB = OutputFolder + "\\" + "Benchmark Tool Results.gdb"
OutputBase = outGDB + "\\" + "benchmarked"
outFC = "Benchmarked_points"

In [26]:
#Create output environment
if os.path.exists(outGDB):
    arcpy.Delete_management(outGDB)
arcpy.CreateFileGDB_management(os.path.dirname(outGDB), os.path.basename(outGDB))

# BM stats array
bmStats = []
bmColumns = ["Objective" , "Benchmark Group", "Condition Rating", "Total Plots", "Total Plots in Group", "Number of Plots Meeting Benchmark", "Percent of Plots Meeting Benchmark"]

# Open the workbook
xl_workbook = xlrd.open_workbook(configFile) # may want to copy this config file to the output excel?
# Assume in the first sheet
xl_sheet = xl_workbook.sheet_by_index(0)

# Field names
row = xl_sheet.row(0)

# Doing all this in_memory
bmOut = "in_memory/BMOut"

arcpy.FeatureClassToFeatureClass_conversion(AnalysisPoints, "in_memory", "BMOut")

totalPlots = int(arcpy.GetCount_management(bmOut)[0])

# perhaps pull this into a data frame for easier calcs...
for rowNum in range(1, xl_sheet.nrows):
    line = xl_sheet.row(rowNum)

    meetingPlots = 0
    groupQuery = "BenchmarkGroup" + '=' +  "'" + line[3].value + "'"
    indicator = line[7].value
    lowerRelation = str(line[6].value)
    lowerLimit = str(line[5].value)
    upperRelation =  str(line[8].value)
    upperLimit = str(line[9].value)
    rule = indicator + " " + lowerRelation + lowerLimit + " " + 'and' + " " + indicator + " " + upperRelation + upperLimit
    outField = indicator + '_benchmarked_' + str(int(line[0].value))
    outAlias = line[1].value + '_' + indicator
    metMessage = "'" + line[11].value + "'"

    # Add the field if it doesnt already exisit and populate it using the query and message
    if arcpy.ListFields(bmOut, outField): #if field exists, evaluates to true
           arcpy.AddMessage("Applying benchmark " + str(rowNum))
    else:
        arcpy.AddField_management(bmOut, outField, "TEXT", field_length=255, field_alias=outAlias)
        arcpy.AddMessage("Applying benchmark " + str(rowNum))

    # Make the view
    arcpy.MakeTableView_management(bmOut, "memView")

    # Run the group query first to get a count for the total
    if groupQuery == "BenchmarkGroup='All'":
        arcpy.SelectLayerByAttribute_management("memView", "SWITCH_SELECTION")
    else:
        arcpy.SelectLayerByAttribute_management("memView", "NEW_SELECTION", groupQuery)

    totalGroupPlots = int(arcpy.GetCount_management("memView")[0])

    # Then do the rule
    arcpy.SelectLayerByAttribute_management("memView", "SUBSET_SELECTION", rule)

    arcpy.CalculateField_management("memView", outField, metMessage, "PYTHON_9.3")
    meetingPlots = int(arcpy.GetCount_management("memView")[0])
    if totalGroupPlots == 0:
        percentMeeting = 0
    else:
        percentMeeting = round((float(meetingPlots) / totalGroupPlots) * 100,2)

#removed notmetmessage here
    arcpy.Delete_management("memView")

    # Add the calcs to list
    bmStats.append(outAlias + "," + line[3].value + "," + metMessage  + "," +  str(totalPlots)  + "," +  str(totalGroupPlots)  + "," + str(meetingPlots)  + "," +  str(percentMeeting))

# Export to Excel
if os.path.exists(outExcel):
    os.remove(outExcel)

arcpy.TableToExcel_conversion(bmOut, outExcel, "ALIAS")

# Export to GDB
arcpy.FeatureClassToFeatureClass_conversion(bmOut, outGDB, outFC)

# Write stats to excel
wb = openpyxl.load_workbook(outExcel)
ws_data = wb.active

# renaming existing sheet
ws_data.title = "Raw Data"

# pull raw data into pandas dataframe, filter to relevant indicators and add conditional formatting for meeting/not meeting
data = ws_data.values
cols = next(data)[1:]
data = list(data)
data = (islice(r, 1, None) for r in data)
df_rawdata = pd.DataFrame(data, columns=cols)

# Select only relevant columns (Indicators) from raw data
# first grab the unique indicators specified in the config file
indicators = xl_sheet.col_values(7)[1:] # removing the first row since thats the header
indicators = list(set(indicators))

# also want the benchmark group column
summary_cols = indicators.copy() # giving this a different name since we want to preserve the list of indicators for plotting later
summary_cols.append("BenchmarkGroup")
summary_cols.append("PrimaryKey")
df_ind_summary = df_rawdata[summary_cols]

# Group by benchmark group
# Summarise (mean, standard error, sample size)
df_ind_summary = df_ind_summary.groupby("BenchmarkGroup").describe(percentiles = []) # this defaults to only summarising the numeric columns

# copy indicator summary to excel sheet
# create new sheet
wb.create_sheet("Indicator Summary")

writer = pd.ExcelWriter(outExcel, engine = 'openpyxl', mode = 'a')
writer.wb = wb

# need to tell excelwriter what sheets already exist
writer.sheets = dict((ws.title, ws) for ws in wb.worksheets)
df_ind_summary.to_excel(writer, sheet_name = "Indicator Summary", index = True)
writer.save()
writer.close()

# adjust formatting of indicator summary sheet
# remove blank cells under headers by merging
# A2:A3 cells are always the same so can hard code
ws_ind_summary = wb["Indicator Summary"]

# we first have to move the header text since openpyxl will only keep the top left cell
ws_ind_summary.move_range('A3', rows = -1)

# then we can just delete the 3rd row
ws_ind_summary.delete_rows(3)

# Creating sheet for Benchmarks summary
ws = wb.create_sheet("Reporting Unit Summary")

# Add column headers
ws.append(bmColumns)

# Add in elements of bmStats to each row
for line in bmStats:
    statsLine = line.split(",")
    ws.append(statsLine)

# change table formatting
# bold headers
for col in range(1, len(bmColumns)+1):
    ws[get_column_letter(col) + '1'].font = Font(bold = True)

# format numbers in cols C:E
for col in range(4,7):
    for row in range(2, ws.max_row+1):
        ws[get_column_letter(col)+ str(row)] = int(ws[get_column_letter(col)+ str(row)].value)

# format percentage in col F
for row in range(2, ws.max_row+1):
    ws['F'+ str(row)] = float(ws['F'+ str(row)].value)

# remove single quotes from second col
for row in range(2, ws.max_row+1):
    cell = ws['B'+ str(row)].value
    ws['B'+ str(row)].value = cell.replace("'", "")

# can also use seaborn
# although this needs an install...


In [25]:
indicators[1]

'SoilStability_Unprotected'

In [27]:

#for i in indicators:
   # for j in 1:len(indicators):
                        
#plt.clf()
df_rawdata.boxplot(column = indicators[1], by = 'BenchmarkGroup') # might actually want a separate plot per benchmark group so we can have benchmarks on there

#plt.savefig(OutputFolder + "/" + indicator[1] + '_boxplot.png')



<AxesSubplot:title={'center':'SoilStability_Unprotected'}, xlabel='BenchmarkGroup'>

In [ ]:
# ideally we should also plot the benchmark line on here
# may want to also just copy this graph to the excel sheet

# first grab the lsit of unique objectives from config file
objectives = xl_sheet.col_values(1)[1:] # removing the first row since thats the header
objectives = list(set(objectives))

# we'll just use the raw data dataframe
# if we want all objectives in same plot well need to pivot the df

#create dataframe from reportunit unit summry tab
data = ws.values
cols = next(data)[1:]
data = list(data)
data = (islice(r, 1, None) for r in data)
df_ru_summary = pd.DataFrame(data, columns=cols)

for objective in objectives:
    for indicator in indicators:
        plt.clf()
        field = objective + '_' + indicator
        data = df_rawdata[field]
        data.value_counts().plot(kind = 'bar')
        plt.savefig(OutputFolder + "/" + objective +  indicator + '_histogram.png')
# need to reorder categories some how...


In [ ]:
# Add plot summary tab
# this should have plot id/primarykey, indicator value, benchmark group, benchmark category

# select columns from raw data
plot_cols = ["PrimaryKey", "PlotID", "BenchmarkGroup"]
plot_cols.extend(indicators)
# also need to add objective category columns to this list
plot_cols.extend(objectives)

df_plotsummary = df_rawdata[plot_cols]

# create new sheet
wb.create_sheet("Plot Summary")

writer2 = pd.ExcelWriter(outExcel, engine = 'openpyxl', mode = 'a')
writer2.wb = wb

# need to tell excelwriter what sheets already exist
writer2.sheets = dict((ws.title, ws) for ws in wb.worksheets)

# write plot summary df to excel
df_plotsummary.to_excel(writer2, sheet_name = "Plot Summary", index = False)
writer2.save()
writer2.close()

# need to widen cols in all sheets
for sheet in wb.worksheets:
    dims = {}
    for row in sheet.rows:
        for cell in row:
            if cell.value:
                dims[cell.column_letter] = max((dims.get(cell.column_letter, 0), len(str(cell.value))))
    for col, value in dims.items():
        sheet.column_dimensions[col].width = value

wb.save(outExcel)